# Formula 1 Podium Prediction — Section 1: Data Audit & Cleaning

**Goal of this section:** turn the 14 raw Ergast CSVs into clean, correctly typed copies that later sections
(data-engineering questions, features, modeling) can trust.

What this section does, in order:

1. **Load** the raw CSVs untouched (raw files stay immutable; we only clean copies).
2. **Audit the raw data:** shapes, the `\N` sentinel, and coverage over the years.
3. **EDA before cleaning** with plots.
4. **Clean:** replace sentinels with real missing values, apply semantic types (dates, lap/pit times → seconds).
5. **Key audit:** primary keys unique & non-null, foreign keys reconcile.
6. **Grain fix:** exactly one row per `(raceId, driverId)` in `results`, with a documented duplicate rule.
7. **EDA after cleaning** and before/after comparison.
8. **Save** the cleaned tables for the next sections.

## 1.0 Setup

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

SENTINEL = "\\N"   # the literal two-character string used for "missing" in these CSVs

DATA_DIR = "/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020"
OUT_DIR = "/kaggle/working/clean" if os.path.isdir("/kaggle/working") else "./clean"
os.makedirs(OUT_DIR, exist_ok=True)
print("Data folder:", DATA_DIR)
print("Output folder:", OUT_DIR)

## 1.1 Load the raw tables (immutable)

We read every file **as text** (`dtype=str`, `keep_default_na=False`) so pandas does not silently
guess types or convert anything. This raw copy is never modified; it is our "before" snapshot.

In [ ]:
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]

raw = {t: pd.read_csv(os.path.join(DATA_DIR, f"{t}.csv"), dtype=str, keep_default_na=False) for t in TABLES}

overview = pd.DataFrame({
    "rows": {t: len(df) for t, df in raw.items()},
    "columns": {t: df.shape[1] for t, df in raw.items()},
}).sort_values("rows", ascending=False)
overview

## 1.2 Raw audit: how much is hidden behind the `\N` sentinel?

Because `\N` is a normal string, a naive `df.isna()` reports **zero** missing values. Counting the sentinel
explicitly shows the true missingness.

In [ ]:
sent_rows = []
for t, df in raw.items():
    for c in df.columns:
        n = (df[c] == SENTINEL).sum()
        if n:
            sent_rows.append({"table": t, "column": c, "n_sentinel": n, "pct": 100 * n / len(df)})
sentinels = pd.DataFrame(sent_rows).sort_values("pct", ascending=False)

print("isna() on the raw data finds", sum(df.isna().sum().sum() for df in raw.values()), "missing values")
print("but there are", sentinels.n_sentinel.sum(), "hidden \\N sentinels in", len(sentinels), "columns")
sentinels.head(25)

In [ ]:
# Total number of columns containing sentinels
print("Columns with sentinels:", len(sentinels))


In [ ]:
top = sentinels.head(50).assign(label=lambda d: d.table + "." + d.column)
fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(top))))
sns.barplot(data=top, y="label", x="pct", color="#4C72B0", ax=ax)
ax.set(title="Share of '\\N' sentinel per column (raw data)", xlabel="% of rows that are '\\N'", ylabel="")
plt.tight_layout(); plt.show()

**Observation to note in the report:** the columns with the most sentinels (e.g. `qualifying.q3`, `results.time`,
`results.fastestLapTime`, the `races.fp*/quali_*/sprint_*` schedule columns) are mostly **structurally missing**:
the value does not exist for that row (a driver knocked out in Q1 has no Q3 time; schedule fields only exist for
recent seasons). They are not data errors, so we convert them to `NaN` rather than dropping rows.

## 1.3 EDA before cleaning

Three questions that matter for a pre-race podium model:

1. How many races and entrants per season? (Field size changes the base podium rate, since there are always 3 podium spots.)
2. **For which years do we actually have qualifying data?** (Our main pre-race signal.)
3. How are raw `grid` values distributed? (Looking for odd codes.)

In [ ]:
races_raw = raw["races"].assign(year=lambda d: d.year.astype(int))
res_raw = raw["results"].merge(races_raw[["raceId", "year"]], on="raceId", how="left")
quali_raw = raw["qualifying"].merge(races_raw[["raceId", "year"]], on="raceId", how="left")

per_year = pd.DataFrame({
    "races": races_raw.groupby("year").raceId.nunique(),
    "races_with_results": res_raw.groupby("year").raceId.nunique(),
    "entries_per_race": res_raw.groupby("year").size() / res_raw.groupby("year").raceId.nunique(),
    "races_with_quali": quali_raw.groupby("year").raceId.nunique(),
}).fillna(0)
per_year["quali_coverage_%"] = 100 * per_year.races_with_quali / per_year.races_with_results.replace(0, np.nan)
per_year["podium_base_rate_%"] = 100 * 3 / per_year.entries_per_race.replace(0, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
per_year[["races", "races_with_results"]].plot(ax=axes[0], title="Races per season")
per_year["entries_per_race"].plot(ax=axes[1], color="C2", title="Average entrants per race")
tw = axes[1].twinx(); tw.plot(per_year.index, per_year["podium_base_rate_%"], color="C3", ls="--")
tw.set_ylabel("podium base rate % (dashed)", color="C3"); tw.grid(False)
axes[1].set_xlabel("year")
per_year["quali_coverage_%"].plot(ax=axes[2], color="C1", title="% of races with a qualifying table")
axes[2].set_ylabel("%")
plt.tight_layout(); plt.show()

first_q = per_year.index[per_year.races_with_quali > 0].min()
full_q = per_year.index[(per_year.index >= first_q) & (per_year["quali_coverage_%"] < 100)].max() + 1
print(f"Qualifying data starts in {first_q}, but is only complete (100% of races) from {full_q} onwards.")
print(per_year.loc[first_q:full_q, ["races_with_results", "races_with_quali", "quali_coverage_%"]].round(0))
print("Races in the calendar with no results:", (per_year.races - per_year.races_with_results).sum())

**Why this matters (hypothesis-driven choices for later sections):**
- The `qualifying` table is empty before the first year printed above and **patchy for several seasons after it**;
  it is only complete from the second year printed. Any model using qualifying features must either be restricted
  to the complete era or handle missing qualifying explicitly. This is the *historical qualifying-coverage drift*
  limitation the brief mentions, and it is a key input for choosing the modeling window.
- Field size changed a lot (dashed red line = 3 / entrants), so the podium base rate is **not constant over time**.
  This supports a **time-based** split and choosing PR-AUC as the primary metric.
- Races present in `races.csv` but without results (future calendar entries) cannot be labeled and are excluded.

In [ ]:
grid_raw = pd.to_numeric(raw["results"]["grid"], errors="coerce")
fig, ax = plt.subplots(figsize=(9, 3.5))
grid_raw.value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set(title="Raw 'grid' values in results", xlabel="grid", ylabel="rows")
ax.tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()
print(f"grid == 0 appears in {(grid_raw == 0).sum()} rows ({100 * (grid_raw == 0).mean():.1f}%).")

# What does grid 0 actually mean? Look at the race status and laps of those rows.
g0 = (raw["results"].merge(raw["status"], on="statusId", how="left")
      .merge(races_raw[["raceId", "year"]], on="raceId", how="left")
      .assign(grid=grid_raw, laps=lambda d: pd.to_numeric(d.laps)))
g0 = g0[g0.grid == 0]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
g0.status.value_counts().head(10).iloc[::-1].plot(kind="barh", ax=axes[0], color="#C44E52")
axes[0].set(title="Status of rows with grid = 0 (top 10)", xlabel="rows")
g0.assign(kind=np.where(g0.laps == 0, "0 laps (never started)", "laps > 0 (raced)")) \
  .pipe(lambda d: pd.crosstab(d.year // 10 * 10, d.kind)).plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set(title="grid = 0 rows by decade", xlabel="decade", ylabel="rows")
plt.tight_layout(); plt.show()
print(f"grid = 0 with 0 laps: {(g0.laps == 0).sum()} | grid = 0 with laps > 0: {(g0.laps > 0).sum()} "
      f"(first year: {g0.loc[g0.laps > 0, 'year'].min()})")

**Finding:** `grid = 0` mixes two very different things:

1. **Drivers who never started the race.** Most `grid = 0` rows have status *Did not qualify*, *Did not prequalify*
   or *Withdrew* and 0 laps. They mostly come from the 1970s–1990s, when more cars entered than the grid allowed.
2. **Genuine pit-lane starts.** A small number of rows (only in the modern era) have `grid = 0` but completed laps.

Treating either as "position 0" would make them look like they started ahead of pole. In cleaning we split them into
two flags: `did_not_start` and `pit_lane_start`.

## 1.4 Cleaning: sentinels and semantic typing

Rules applied to **copies** of the raw tables:

| Step | What | Why |
|---|---|---|
| Sentinel | `\N` → `NaN` in every column | so pandas counts and handles missingness correctly |
| Numeric | IDs, positions, points, laps, ms → numbers | they were read as text |
| Dates | `races.date`, `drivers.dob`, schedule dates → `datetime` | allows date maths (age at race, ordering) |
| Durations | `q1/q2/q3`, lap times, pit `duration`, `fastestLapTime` → **seconds** (float) | `"1:23.456"` cannot be compared or averaged as text |
| Grid 0 / non-starters | flags `did_not_start` and `pit_lane_start`; `grid` set to `NaN` when 0 | 0 is a code, not a position |

Kept as text on purpose: `positionText` (`R` = retired, `D` = disqualified, … are codes, not missing values) and
`results.time` (it mixes the winner's total time with `+gap` strings; `milliseconds` holds the usable number).

In [ ]:
def time_to_seconds(x):
    # '1:23.456' -> 83.456, '23.456' -> 23.456, '1:02:03.4' -> 3723.4, missing -> NaN
    if pd.isna(x) or x == "":
        return np.nan
    try:
        parts = [float(p) for p in str(x).split(":")]
    except ValueError:
        return np.nan
    secs = 0.0
    for p in parts:
        secs = secs * 60 + p
    return secs

# quick self-test
assert time_to_seconds("1:23.456") == 83.456
assert time_to_seconds("23.456") == 23.456
assert time_to_seconds("31:40.500") == 1900.5
assert np.isnan(time_to_seconds(np.nan))
print("time_to_seconds OK")

In [ ]:
pd.DataFrame({"raw dtype": raw["races"].dtypes.astype(str)})

In [ ]:
pd.DataFrame({"raw dtype": raw["results"].dtypes.astype(str)})

In [ ]:
pd.DataFrame({"raw dtype": raw["qualifying"].dtypes.astype(str)})

To make sure which types are not numeric, turned out to be all


In [ ]:
NUMERIC = {
    "circuits": ["circuitId", "lat", "lng", "alt"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId", "raceId", "constructorId", "points"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "points", "position", "wins"],
    "drivers": ["driverId", "number"],
    "driver_standings": ["driverStandingsId", "raceId", "driverId", "points", "position", "wins"],
    "races": ["raceId", "year", "round", "circuitId"],
    "results": ["resultId", "raceId", "driverId", "constructorId", "number", "grid", "position", "positionOrder",
                "points", "laps", "milliseconds", "fastestLap", "rank", "fastestLapSpeed", "statusId"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "number", "position"],
    "lap_times": ["raceId", "driverId", "lap", "position", "milliseconds"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],
    "sprint_results": ["resultId", "raceId", "driverId", "constructorId", "number", "grid", "position",
                       "positionOrder", "points", "laps", "milliseconds", "fastestLap", "statusId"],
    "seasons": ["year"],
    "status": ["statusId"],
}
DATES = {"races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"], "drivers": ["dob"]}
DURATIONS = {"qualifying": ["q1", "q2", "q3"], "lap_times": ["time"], "pit_stops": ["duration"],
             "results": ["fastestLapTime"], "sprint_results": ["fastestLapTime"]}

clean = {}
for t, df in raw.items():
    d = df.copy()                                  # never touch the raw copy
    d = d.replace({SENTINEL: np.nan, "": np.nan})  # 1) sentinel -> NaN
    for c in NUMERIC.get(t, []):                   # 2) numbers
        if c in d:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    for c in DATES.get(t, []):                     # 3) dates
        if c in d:
            d[c] = pd.to_datetime(d[c], errors="coerce")
    for c in DURATIONS.get(t, []):                 # 4) durations -> seconds (new *_sec column, original kept)
        if c in d:
            d[c + "_sec"] = d[c].map(time_to_seconds)
    clean[t] = d

# 5) Non-starters and pit-lane starts (done after the check, because it creates NaNs on purpose)
#    did_not_start: the driver never took the start. Either the status is a pre-race elimination
#    (DNQ / DNPQ / 107% rule) or the car has grid 0 AND completed 0 laps.
#    This is known before lights out (it is decided in qualifying), so filtering on it later is not leakage.
PRE_RACE_OUT = ["Did not qualify", "Did not prequalify", "107% Rule"]
pre_race_ids = clean["status"].loc[clean["status"].status.isin(PRE_RACE_OUT), "statusId"]
for t in ["results", "sprint_results"]:
    d = clean[t]
    d["did_not_start"] = (d.statusId.isin(pre_race_ids) | ((d.grid == 0) & (d.laps == 0))).astype(int)
    d["pit_lane_start"] = ((d.grid == 0) & (d.did_not_start == 0)).astype(int)
    d.loc[d.grid == 0, "grid"] = np.nan

r_ = clean["results"]
print(f"results: did_not_start = {r_.did_not_start.sum()} rows "
      f"(podiums among them: {((r_.did_not_start == 1) & (r_.positionOrder <= 3)).sum()}), "
      f"pit_lane_start = {r_.pit_lane_start.sum()} rows")

In [ ]:
clean["qualifying"].head


In [ ]:
# Before vs after: dtypes of the results table
pd.DataFrame({"raw dtype": raw["results"].dtypes.astype(str), "clean dtype": clean["results"].dtypes.astype(str)})

### Sanity checks on the converted durations

Converting to seconds lets us check whether values are physically plausible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
q = clean["qualifying"][["q1_sec", "q2_sec", "q3_sec"]].melt(var_name="session", value_name="seconds").dropna()
sns.histplot(data=q, x="seconds", hue="session", bins=60, element="step", ax=axes[0])
axes[0].set_title("Qualifying lap times (s)")
sns.histplot(clean["lap_times"]["time_sec"].dropna(), bins=80, ax=axes[1], color="C2")
axes[1].set(title="Race lap times (s)", yscale="log")
sns.histplot(clean["pit_stops"]["duration_sec"].dropna(), bins=80, ax=axes[2], color="C3")
axes[2].set(title="Pit-stop durations (s)", yscale="log")
plt.tight_layout(); plt.show()

ps = clean["pit_stops"]["duration_sec"]
print(f"Pit stops longer than 60 s: {(ps > 60).sum()} of {ps.notna().sum()} "
      f"(red-flag periods are recorded as very long 'stops', e.g. {ps.max():.0f} s)")
lt = clean["lap_times"]
print("lap_times: 'time' and 'milliseconds' agree:",
      np.allclose(lt["time_sec"], lt["milliseconds"] / 1000, atol=0.01, equal_nan=True))

**Decision:** extreme pit-stop and lap-time values (red flags, safety-car laps) are **real events, not errors**,
so we keep them. They only matter for the data-engineering questions; pit stops and lap times are *post-race*
information and are **not allowed as model features** (leakage).

## 1.5 Key audit: primary keys and foreign keys

- **Primary key (PK):** must be unique and never missing.
- **Foreign key (FK):** every value must exist in the table it points to.

In [ ]:
PKS = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"], "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"], "races": ["raceId"], "results": ["resultId"],
    "qualifying": ["qualifyId"], "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"], "seasons": ["year"], "status": ["statusId"],
}
pk_report = pd.DataFrame([{
    "table": t, "primary_key": "+".join(k),
    "null_keys": int(clean[t][k].isna().any(axis=1).sum()),
    "duplicate_keys": int(clean[t].duplicated(subset=k).sum()),
} for t, k in PKS.items()])
pk_report["ok"] = (pk_report.null_keys == 0) & (pk_report.duplicate_keys == 0)
pk_report

In [ ]:
FKS = [  # (child table, column, parent table, parent column)
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    *[(t, "raceId", "races", "raceId") for t in ["results", "qualifying", "driver_standings", "constructor_standings",
                                                  "constructor_results", "lap_times", "pit_stops", "sprint_results"]],
    *[(t, "driverId", "drivers", "driverId") for t in ["results", "qualifying", "driver_standings", "lap_times",
                                                      "pit_stops", "sprint_results"]],
    *[(t, "constructorId", "constructors", "constructorId") for t in ["results", "qualifying", "constructor_standings",
                                                                     "constructor_results", "sprint_results"]],
    ("results", "statusId", "status", "statusId"), ("sprint_results", "statusId", "status", "statusId"),
]
fk_rows = []
for child, col, parent, pcol in FKS:
    vals = clean[child][col].dropna()
    orphans = ~vals.isin(clean[parent][pcol])
    fk_rows.append({"fk": f"{child}.{col} → {parent}.{pcol}", "rows": len(vals),
                    "orphans": int(orphans.sum()), "example_orphans": list(vals[orphans].unique()[:5])})
fk_report = pd.DataFrame(fk_rows)
fk_report["ok"] = fk_report.orphans == 0

fk_report

## 1.6 Grain: one row per `(raceId, driverId)`

Each modeling row is one prediction ("will this driver podium in this race?"), so `(raceId, driverId)` must be unique.

// These are the 4 tables with both raceID & DriverID


In [ ]:
grain_tables = ["results", "qualifying", "driver_standings", "sprint_results"]
grain = pd.DataFrame([{
    "table": t,
    "duplicate (raceId, driverId) rows": int(clean[t].duplicated(["raceId", "driverId"], keep=False).sum()),
} for t in grain_tables])
grain

In [ ]:
res = clean["results"].merge(clean["races"][["raceId", "year", "name"]], on="raceId", how="left")
dup_mask = res.duplicated(["raceId", "driverId"], keep=False)
dups = res[dup_mask].sort_values(["raceId", "driverId", "resultId"])
print(f"{dups[['raceId', 'driverId']].drop_duplicates().shape[0]} driver-race pairs are duplicated "
      f"({dup_mask.sum()} rows) across {dups.raceId.nunique()} races.")

if len(dups):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    dups.drop_duplicates(["raceId", "driverId"]).year.value_counts().sort_index().plot(kind="bar", ax=ax, color="#C44E52")
    ax.set(title="Duplicated driver-race pairs by season (shared drives)", xlabel="year", ylabel="pairs")
    plt.tight_layout(); plt.show()

dups[["raceId", "year", "name", "driverId", "constructorId", "grid", "positionOrder", "laps", "statusId", "points"]].head(12)

In [ ]:
def keep_most_laps(df):
    return df.sort_values(["laps", "resultId"], ascending=[False, True]).drop_duplicates(["raceId", "driverId"])
def keep_best_finish(df):
    return df.sort_values(["positionOrder", "resultId"]).drop_duplicates(["raceId", "driverId"])
def keep_first(df):
    return df.sort_values("resultId").drop_duplicates(["raceId", "driverId"])

cmp = pd.DataFrame([{
    "rule": name, "rows": len(f(res)),
    "podium rows kept": int((f(res).positionOrder <= 3).sum()),
    "podium rows among the duplicated pairs": int((f(res[dup_mask]).positionOrder <= 3).sum()),
} for name, f in [("Most laps", keep_most_laps), ("Best finish", keep_best_finish), ("First row", keep_first)]])
cmp

Rule — keep the row with the most laps, and if there are 2 entries with the same lap count, we take the first row.
Any duplicates found in `qualifying`, `driver_standings` or `sprint_results` (table above) are handled with the
same idea: keep one row per `(raceId, driverId)`.

In [ ]:
before = len(clean["results"])
clean["results"] = keep_most_laps(clean["results"]).sort_values("resultId").reset_index(drop=True)
print(f"results: {before} -> {len(clean['results'])} rows")

for t in ["qualifying", "driver_standings", "sprint_results"]:
    n = clean[t].duplicated(["raceId", "driverId"]).sum()
    if n:
        sort_col = {"qualifying": "position", "driver_standings": "driverStandingsId", "sprint_results": "resultId"}[t]
        clean[t] = clean[t].sort_values(sort_col).drop_duplicates(["raceId", "driverId"]).reset_index(drop=True)
        print(f"{t}: removed {n} duplicate rows")

for t in grain_tables:
    assert not clean[t].duplicated(["raceId", "driverId"]).any(), t
print("Grain check passed: (raceId, driverId) is unique in", ", ".join(grain_tables))

## 1.7 EDA after cleaning: before vs after

In [ ]:
def missing_pct(df_dict, is_raw):
    rows = []
    for t in ["results", "qualifying", "races", "drivers", "pit_stops", "driver_standings"]:
        d = df_dict[t]
        d = d[raw[t].columns]   # compare the same original columns (ignore the new *_sec / flag columns)
        miss = (d == SENTINEL) | d.isna() if is_raw else d.isna()
        rows.append({"table": t, "% cells missing": 100 * miss.values.mean()})
    return pd.DataFrame(rows)

mb = missing_pct(raw, True).assign(stage="what isna() sees on raw").assign(**{"% cells missing": lambda d: 0.0})
mt = missing_pct(raw, True).assign(stage="true missing (raw, \\N counted)")
ma = missing_pct(clean, False).assign(stage="isna() after cleaning")
cmp_miss = pd.concat([mb, mt, ma])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.barplot(data=cmp_miss, x="table", y="% cells missing", hue="stage", ax=axes[0])
axes[0].set_title("Missingness: hidden before cleaning, visible after")
axes[0].tick_params(axis="x", rotation=20)

r = clean["results"].merge(clean["races"][["raceId", "year"]], on="raceId")
r["podium"] = (r.positionOrder <= 3).astype(int)
sns.boxplot(data=r, x="podium", y="grid", ax=axes[1])
axes[1].set(title="Grid position vs podium (pit-lane starts excluded)", xticklabels=["no podium", "podium"])
plt.tight_layout(); plt.show()

starters = r[r.did_not_start == 0]
print(f"Podium rate, all rows: {100 * r.podium.mean():.1f}% | among drivers who started: "
      f"{100 * starters.podium.mean():.1f}%  -> rare positive class")
print(f"Pit-lane starts: {int(r.pit_lane_start.sum())} | podiums from the pit lane: "
      f"{int(r.loc[r.pit_lane_start == 1, 'podium'].sum())}")

In [ ]:
# Duration columns: text before cleaning, numeric after
ex = clean["qualifying"][["q1", "q1_sec", "q2", "q2_sec", "q3", "q3_sec"]].dropna().head(5)
ex

**Summary of the cleaning effects**

- Missing values were hidden as `\N` text; after cleaning they are real `NaN`s and appear in `isna()`.
- Dates are `datetime` and all lap/qualifying/pit times are numeric seconds (`*_sec` columns).
- `grid = 0` is split into **non-starters** (`did_not_start`, mostly DNQ/DNPQ) and genuine **pit-lane starts**
  (`pit_lane_start`); neither looks like "ahead of pole" any more. The modeling table should keep only starters.
- PKs are unique and non-null, FKs reconcile (or orphans were logged and dropped).
- `results` now has exactly one row per `(raceId, driverId)`.
- The grid-vs-podium boxplot already shows a strong separation, which motivates grid/qualifying features later.

## 1.8 Save the cleaned tables

In [ ]:
for t, d in clean.items():
    d.to_pickle(os.path.join(OUT_DIR, f"{t}.pkl"))   # pickle keeps dtypes (dates, floats) exactly
print("Saved", len(clean), "clean tables to", OUT_DIR)
print("Load later with: pd.read_pickle(f'{OUT_DIR}/results.pkl')")

# Section 2: Data-Engineering Questions

Each question is answered with **multi-hop joins** on the cleaned tables from Section 1, a **plot**, and a short
note on **grain**, **missingness** and the **pre-race cutoff**.

All three questions use one row per **(raceId, driverId)** and only drivers who **started** the race
(`did_not_start == 0` from Section 1).

In [ ]:
base = (clean["results"]
        .merge(clean["races"][["raceId", "year", "circuitId"]], on="raceId")
        .merge(clean["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit"}), on="circuitId")
        .merge(clean["status"][["statusId", "status"]], on="statusId"))
base["podium"] = (base.positionOrder <= 3).astype(int)
starts = base[base.did_not_start == 0].copy()
print("Race starts:", len(starts))

## Q1. Which circuits convert a front-row start into a podium most often, and does that ranking change after 2014?

**Joins:** `results` ⋈ `races` ⋈ `circuits` ⋈ `qualifying`

- **Front row by grid** = started from grid 1–2. **Front row by qualifying** = qualified P1–2.
  They can differ because of grid penalties.
- **Conversion rate** = podiums ÷ front-row starts.
- **Eras:** 2003–2013 vs 2014–2024 (from 2003 because qualifying data is only complete from 2003, see Section 1).
- A circuit needs **at least 6 front-row starts** (about 3 races) in an era to be ranked, so a circuit with one
  race cannot top the list by luck.

In [ ]:
quali = clean["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "quali_pos"})
q1 = starts[starts.year >= 2003].merge(quali, on=["raceId", "driverId"], how="left")
q1["era"] = np.where(q1.year >= 2014, "2014–2024", "2003–2013")

# Qualifying position vs grid
grid_front = q1[q1.grid <= 2]
quali_front = q1[q1.quali_pos <= 2]
pd.DataFrame({
    "front row by grid: starts": grid_front.groupby("era").size(),
    "front row by grid: podium rate": grid_front.groupby("era").podium.mean(),
    "front row by qualifying: starts": quali_front.groupby("era").size(),
    "front row by qualifying: podium rate": quali_front.groupby("era").podium.mean(),
    "qualified P1–2 but started lower (penalty)": quali_front[~(quali_front.grid <= 2)].groupby("era").size(),
}).T.round(2)

In [ ]:
# Conversion rate per circuit (grid definition)
conv = grid_front.groupby(["circuit", "era"]).podium.agg(front_row_starts="size", rate="mean").reset_index()
conv = conv[conv.front_row_starts >= 6]

ranking = conv.pivot(index="circuit", columns="era", values="rate").dropna()
ranking["rank 2003–13"] = ranking["2003–2013"].rank(ascending=False).astype(int)
ranking["rank 2014–24"] = ranking["2014–2024"].rank(ascending=False).astype(int)
print("Rank correlation between eras:", round(ranking["2003–2013"].corr(ranking["2014–2024"], method="spearman"), 2),
      "(1 = same ranking, 0 = unrelated)")
ranking.sort_values("rank 2014–24").round(2)

In [ ]:
ranking.sort_values("2014–2024")[["2003–2013", "2014–2024"]].plot(kind="barh", figsize=(9, 7))
plt.xlabel("front-row → podium conversion rate")
plt.title("Front-row conversion by circuit, 2003–13 vs 2014–24")
plt.xlim(0, 1); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

### Answer to Q1

**Qualifying vs grid:**
- Front-row starters by grid reached the podium **68%** of the time (2003–13) and **76%** (2014–24).
- By qualifying position the rates are almost the same: **67%** and **74%**.
- Penalties moved **9** front-row qualifiers further back before 2014 and **27** after, so the two definitions differ more in the hybrid era.

**Best circuits:**
- **2003–13:** Monaco and Monza (77%), then Silverstone (73%).
- **2014–24:** Suzuka (89%), Yas Marina (86%), Sepang (86%), Monaco (85%).

**Did the ranking change after 2014? Yes.**
- The rank correlation between the two eras is only **0.16**, so the order of circuits is very different.
- Yas Marina jumped from 13th to 2nd.
- Monza dropped from 1st to 12th.
- Conversion also went up overall (68% → 76%), because dominant cars usually started at the front and stayed there.

**Grain:** one row per driver per race start.

**Missingness:**
- Only 2003+ is used, because qualifying data is incomplete before that.
- Drivers with no grid slot (pit-lane starts) or no qualifying time count as "not front row".

**Pre-race cutoff:** grid and qualifying position are both known before the race starts.

## Q2. Do drivers podium more often at home, after controlling for starting position?

**Joins:** `results` ⋈ `drivers` ⋈ `races` ⋈ `circuits`

- **Home race** = driver nationality matches the circuit's country. Nationality is "British" but country is "UK",
  so we map one to the other.
- **Controlling for starting position:** we compare home vs away drivers **inside the same grid group**
  (P1, P2–3, P4–6, P7–10, P11+).

In [ ]:
NAT_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina", "Argentine-Italian": "Argentina",
    "Argentinian": "Argentina", "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "UK", "Canadian": "Canada", "Chinese": "China", "Dutch": "Netherlands",
    "East German": "Germany", "French": "France", "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Italian": "Italy", "Japanese": "Japan", "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "Portuguese": "Portugal", "Russian": "Russia", "South African": "South Africa", "Spanish": "Spain",
    "Swedish": "Sweden", "Swiss": "Switzerland",
}
h = starts.merge(clean["drivers"][["driverId", "nationality"]], on="driverId")
h["home"] = (h.nationality.str.strip().map(NAT_TO_COUNTRY) == h.country.replace({"United States": "USA"})).astype(int)
h = h[h.grid.notna()]
h["grid_group"] = pd.cut(h.grid, [0, 1, 3, 6, 10, 100], labels=["P1", "P2–3", "P4–6", "P7–10", "P11+"])

print("Without controlling for grid:")
print(h.groupby("home").agg(podium_rate=("podium", "mean"), average_grid=("grid", "mean"))
        .rename(index={0: "away", 1: "home"}).round(3))

by_grid = h.pivot_table(index="grid_group", columns="home", values="podium", aggfunc=["mean", "size"], observed=True)
by_grid.columns = ["away podium rate", "home podium rate", "away starts", "home starts"]
by_grid.round(3)

In [ ]:
by_grid[["away podium rate", "home podium rate"]].plot(kind="bar", figsize=(9, 4), color=["C0", "C3"], rot=0)
plt.ylabel("podium rate"); plt.xlabel("starting grid group")
plt.title("Podium rate at home vs away, for drivers starting from similar positions")
plt.tight_layout(); plt.show()

### Answer to Q2

**No. Drivers do not podium more often at home once starting position is controlled for.**

- **Without control, home drivers look *worse*:** 11.5% podium rate vs 13.8% away. That's because home drivers start further back on average (grid 13.4 vs 11.7).
- **For drivers starting from similar positions, home and away rates are almost the same:**
  - P2–3: 54% at home vs 51% away
  - P4–6: 25% vs 26%
  - P7–10: 9% vs 9%
  - P11+: 2% vs 2%
- The differences go in both directions and are small, so there is no home advantage.

**Grain:** one row per driver per race start with a known grid position.

**Missingness:**
- Pit-lane starts are left out because they have no grid position.
- Drivers from countries with no F1 circuit are always "away".

**Pre-race cutoff:** nationality, circuit country and grid position are all known before the race.

## Q3. Which constructors have the highest mechanical-retirement rate in 2014–2021 vs 2022–2024?

**Joins:** `results` ⋈ `status` ⋈ `races` ⋈ `constructors`

- **Mechanical retirement** = a race start that ended early because a car part failed. We list these statuses by
  hand. Crashes (Accident, Collision, Spun off …), driver causes (Illness) and regulation decisions
  (Disqualified, Withdrew …) are **not** mechanical, as the brief says. Statuses that do not say what failed
  ("Retired", punctures, wings) are also not counted.
- **Rate** = mechanical retirements ÷ race starts. A constructor needs **at least 30 starts** in an era to be ranked.

In [ ]:
MECHANICAL = [
    "Engine", "Gearbox", "Transmission", "Clutch", "Hydraulics", "Electrical", "Radiator", "Suspension", "Brakes",
    "Differential", "Overheating", "Mechanical", "Driveshaft", "Fuel pressure", "Water pressure", "Throttle",
    "Steering", "Technical", "Electronics", "Exhaust", "Oil leak", "Water leak", "Fuel pump", "Oil pressure",
    "Power loss", "Vibrations", "Drivetrain", "Battery", "Turbo", "Water pump", "Spark plugs", "Fuel leak",
    "Power Unit", "ERS", "Brake duct", "Seat", "Cooling system",
]
m = (starts[starts.year.between(2014, 2024)]
     .merge(clean["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId"))
m["era"] = np.where(m.year <= 2021, "2014–2021", "2022–2024")
m["mechanical"] = m.status.isin(MECHANICAL).astype(int)

print("Not counted as mechanical:",
      sorted(set(m.status[(m.mechanical == 0) & (m.status != "Finished") & ~m.status.str.startswith("+")])))

rates = m.groupby(["era", "constructor"]).agg(starts=("mechanical", "size"), mechanical=("mechanical", "sum"))
rates["rate"] = rates.mechanical / rates.starts
rates = rates[rates.starts >= 30].reset_index()

for era in ["2014–2021", "2022–2024"]:
    print(f"\n{era}:")
    print(rates[rates.era == era].sort_values("rate", ascending=False).round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)
for ax, era, color in zip(axes, ["2014–2021", "2022–2024"], ["C0", "C1"]):
    e = rates[rates.era == era].sort_values("rate")
    ax.barh(e.constructor, e.rate, color=color)
    ax.set(title=f"Mechanical-retirement rate, {era}", xlabel="mechanical retirements / race starts")
plt.tight_layout(); plt.show()

### Answer to Q3

**Highest mechanical-retirement rate:**
- **2014–2021:** Caterham **23.5%** (8 of 34 starts), Lotus F1 17.1%, Toro Rosso 16.1%, McLaren 13.8%, Renault 13.5%.
- **2022–2024:** Alfa Romeo **12.5%** (11 of 88), Alpine 11.0%, AlphaTauri 6.8%.

**Lowest:**
- **2014–2021:** Aston Martin, Alfa Romeo and Mercedes, all under 4%.
- **2022–2024:** Sauber (2.1%), then Red Bull, McLaren and Aston Martin (3.7%).

**Overall:** rates are lower in 2022–2024, so cars became more reliable. McLaren went from one of the worst to one of the best.

**Grain:** one row per driver per race start, 2014–2024.

**Missingness:**
- Every race start has a status.
- Statuses that don't say what failed ("Retired", punctures) are not counted as mechanical.
- Renamed teams (e.g. Toro Rosso → AlphaTauri) count as separate constructors.

**Pre-race cutoff:** the retirement reason is only known **after** the race, so it can never be used as a feature for that same race.

# Section 3: Feature Selection & Engineering

**Goal:** build the modeling table: **one row per (raceId, driverId)**, target `podium` (positionOrder ≤ 3),
using only information known **after qualifying and before the race starts**.

**Rows used:**
- Drivers who **started** the race (`did_not_start == 0`).
- Seasons **2003–2024**, because Section 1 showed qualifying data is only complete from 2003.

**Features (3 groups):**

| Group | Feature | Pre-race? |
|---|---|---|
| Qualifying | `grid`, `quali_position` | ✅ set by qualifying (and penalties) before the race |
| Standings | `driver_points_before`, `driver_rank_before`, `constructor_points_before`, `constructor_rank_before` | ✅ taken from the **previous** race of the season |
| Form | `driver_form` (podium rate, last 5 races), `constructor_form` (team podiums per race, last 5 races, 0–2) | ✅ computed from **past races only** |

**Not used (leakage):** this race's `position`, `points`, `laps`, `status`, `milliseconds`, fastest lap, pit stops,
lap times, and the standings **of this race** (they already include this race's result).

## 3.1 Leakage-safe features

**Standings:** in `driver_standings` / `constructor_standings`, the row for a race shows the standings **after**
that race. So for each race we use the standings of the **previous race of the same season**.
In round 1 nobody has points yet, so points = 0 and rank is unknown (missing). After round 1, a driver or team that
is missing from the previous standings had 0 points, so its rank is set to last place in that race.

**Form:** a driver's podium rate over his **previous 5 races in the same season** (`shift(1)` removes the current
race before the rolling average). For the constructor we first count the team's podiums per race, then average the
previous 5 races of the same season, so a teammate's result in the *same* race is never used. Form restarts every
season, so in a driver's or team's first race of a season there is no form yet (missing, filled with the median in 3.4).

In [ ]:
clean["results"].head

In [ ]:
races = clean["races"][["raceId", "year", "round", "date"]].sort_values(["year", "round"])

# All race starts in history (needed to compute form from past races)
hist = clean["results"].merge(races, on="raceId")
hist = hist[hist.did_not_start == 0].sort_values(["date", "raceId"]).copy()
hist["podium"] = (hist.positionOrder <= 3).astype(int)

# Form: driver podium rate in his previous 5 races (shift(1) = exclude the current race)
hist["driver_form"] = hist.groupby(["driverId", "year"]).podium.transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

# Form: constructor podiums per race, averaged over its previous 5 races
team = hist.groupby(["constructorId", "year", "raceId", "date"], as_index=False).podium.sum().sort_values(["date", "raceId"])
team["constructor_form"] = team.groupby(["constructorId", "year"]).podium.transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

# Standings: use the PREVIOUS race of the same season
races["prev_raceId"] = races.groupby("year").raceId.shift(1)
ds = clean["driver_standings"][["raceId", "driverId", "points", "position"]].rename(columns={
    "raceId": "prev_raceId", "points": "driver_points_before", "position": "driver_rank_before"})
cs = clean["constructor_standings"][["raceId", "constructorId", "points", "position"]].rename(columns={
    "raceId": "prev_raceId", "points": "constructor_points_before", "position": "constructor_rank_before"})
ds["prev_raceId"] = ds.prev_raceId.astype(float)
cs["prev_raceId"] = cs.prev_raceId.astype(float)

quali = clean["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})

df = (hist[hist.year >= 2003]
      .merge(races[["raceId", "prev_raceId"]], on="raceId")
      .merge(team[["raceId", "constructorId", "constructor_form"]], on=["raceId", "constructorId"], how="left")  # FIX: bring constructor_form into df
      .merge(ds, on=["prev_raceId", "driverId"], how="left")
      .merge(cs, on=["prev_raceId", "constructorId"], how="left")
      .merge(quali, on=["raceId", "driverId"], how="left"))

# No points before the first race of a season (or before a driver's first race of the season)
df[["driver_points_before", "constructor_points_before"]] = df[["driver_points_before", "constructor_points_before"]].fillna(0)

# Missing rank after round 1 = driver/team not in the previous standings (0 points) -> put them in last place.
# In round 1 nobody has a rank, so max is NaN and those rows stay missing (filled with the median in 3.4).
for col in ["driver_rank_before", "constructor_rank_before"]:
    df[col] = df[col].fillna(df.groupby("raceId")[col].transform("max") + 1)

# Pit-lane starters have no grid number: they start from the back, so grid = number of starters in that race
df["grid"] = df.grid.fillna(df.groupby("raceId").driverId.transform("size"))

QUALIFYING = ["grid", "quali_position"]
STANDINGS = ["driver_points_before", "driver_rank_before", "constructor_points_before", "constructor_rank_before"]
FORM = ["driver_form", "constructor_form"]
FEATURE_GROUPS = {"qualifying": QUALIFYING, "standings": STANDINGS, "form": FORM}
FEATURES = QUALIFYING + STANDINGS + FORM
TARGET = "podium"

assert not df.duplicated(["raceId", "driverId"]).any()
print("Modeling table:", df.shape[0], "rows (one per driver per race),", len(FEATURES), "features")
print("Podium rate:", round(df.podium.mean(), 3))
df[["year", "round", "driverId", "constructorId"] + FEATURES + [TARGET]].head(50)

In [ ]:
# Leakage checks
r1 = df[df["round"] == 1]
assert (r1.driver_points_before == 0).all() and r1.driver_rank_before.isna().all()
print("Round 1: all drivers have 0 points before the race and no rank -> standings come from the previous race.")

# Example: Lewis Hamilton (driverId 1), first 8 races of 2016. driver_form uses only the races BEFORE each row.
ex = df[(df.driverId == 1) & (df.year == 2016)].sort_values("round").head(8)
ex[["year", "round", "podium", "driver_form"]]

In the example above, `driver_form` in each row is the average of the `podium` values of the driver's previous
5 races **of the same season**, never including the row itself. Round 1 has no earlier race in 2016, so its form is
missing. This confirms the form feature uses past races only.

## 3.2 Train / validation / test split

We split **by time**, not randomly:

| Split | Seasons | Use |
|---|---|---|
| Train | 2003–2017 | fit the models |
| Validation | 2018–2020 | choose hyperparameters and the decision threshold |
| Test | 2021–2024 | final, unseen evaluation (used once) |

**Why by time:** the model will be used to predict **future** races. A random split would let the model learn from
races that happen *after* the ones it is tested on (e.g. train on 2023 and test on 2019), and rows from the same race
would end up in different splits. A time split copies real use: train on the past, predict the future.

In [ ]:
df["split"] = np.select([df.year <= 2017, df.year <= 2020], ["train", "val"], default="test")
train, val, test = df[df.split == "train"], df[df.split == "val"], df[df.split == "test"]

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

split_info = df.groupby("split").agg(seasons=("year", lambda y: f"{y.min()}–{y.max()}"), rows=("podium", "size"),
                                     races=("raceId", "nunique"), podium_rate=("podium", "mean")).loc[["train", "val", "test"]]
split_info["share_of_rows"] = split_info.rows / split_info.rows.sum()
split_info.round(3)

## 3.3 Evidence for the chosen features

All evidence below uses the **training seasons only**, so the test set is not looked at while choosing features.

**Hypothesis from the earlier EDA:** Section 1 showed podium finishers start much further forward (grid boxplot),
and Section 2 showed front-row starters reach the podium about 70% of the time. So qualifying/grid should be the
strongest group. Standings and form describe how strong the driver and car have been **so far**, which grid alone
may not fully capture.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, f in zip(axes.ravel(), FEATURES):
    binned = pd.qcut(train[f], q=8, duplicates="drop")
    rate = train.groupby(binned, observed=True).podium.mean()
    ax.plot(range(len(rate)), rate.values, marker="o")
    ax.set_xticks(range(len(rate)), [f"{iv.right:.2g}" for iv in rate.index], rotation=45, fontsize=8)
    ax.set(title=f, ylabel="podium rate", xlabel="bin upper edge", ylim=(0, 1))
plt.suptitle("Podium rate across the values of each feature (training data)")
plt.tight_layout(); plt.show()

In [ ]:
corr = train[FEATURES + [TARGET]].corr(method="spearman")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr[TARGET].drop(TARGET).sort_values().plot(kind="barh", ax=axes[0], color="C0")
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set(title="Correlation of each feature with podium (Spearman, training data)", xlabel="correlation")
sns.heatmap(corr.drop(index=TARGET, columns=TARGET), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Correlation between features")
plt.tight_layout(); plt.show()
corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3)

**What the evidence shows:**

- **Every feature has a clear, one-directional relationship with podium** (line plots):
  - Starting P1–3 gives a **63%** podium rate, P4–6 about 23%, and beyond P10 almost 0%.
  - More points, a better championship rank and better recent form all raise the podium rate steadily.
- **All 8 features are strongly correlated with podium**, with Spearman values between 0.36 and 0.51:
  - form: 0.51 / 0.51
  - qualifying and grid: −0.49 (a lower position number is better)
  - championship rank: −0.47 / −0.44
  - points: 0.37 / 0.36
- **Some features overlap with each other** (heatmap):
  - `grid` vs `quali_position`: 0.97
  - points vs rank: −0.86 (driver) and −0.82 (constructor)
  - We still keep both in each pair:
    - Grid and qualifying differ when there are penalties (Section 2, Q1).
    - Points show *how big* the gap is, while rank only shows the order.
  - The ablation in Section 5 measures how much each group really adds.

**Feature left out on purpose:** a "home race" flag. It is a valid pre-race feature, but Section 2 (Q2) showed no home advantage once grid position is taken into account.

## 3.4 Pre-processing and its effect on the data

Two steps, both **fitted on the training data only** and then applied to validation and test:

1. **Imputation (median):** fills missing values. Missing values come from: `driver_rank_before` /
   `constructor_rank_before` in round 1 (no standings exist yet), `quali_position` when a driver has no qualifying
   record, and `driver_form` / `constructor_form` in a driver's or team's first race of each season.
2. **Standard scaling:** puts every feature on mean 0, standard deviation 1. Points go up to ~700 while form is
   between 0 and 2. Logistic regression and the neural network are sensitive to these different ranges;
   tree models are not affected.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def make_preprocessor():
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])

missing = pd.DataFrame({"train": X_train.isna().sum(), "val": X_val.isna().sum(), "test": X_test.isna().sum()})
print("Missing values per feature BEFORE pre-processing:")
display(missing[missing.sum(axis=1) > 0])

prep = make_preprocessor().fit(X_train)
X_train_p = pd.DataFrame(prep.transform(X_train), columns=FEATURES, index=X_train.index)
print("Missing values AFTER pre-processing (train):", int(X_train_p.isna().sum().sum()))

before_after = pd.concat({"before": X_train.describe().T[["mean", "std", "min", "max"]],
                          "after": X_train_p.describe().T[["mean", "std", "min", "max"]]}, axis=1)
before_after.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
X_train[["grid", "driver_points_before", "constructor_points_before", "driver_form"]].plot(
    kind="box", ax=axes[0], title="Before: very different ranges (training data)")
X_train_p[["grid", "driver_points_before", "constructor_points_before", "driver_form"]].plot(
    kind="box", ax=axes[1], title="After imputation + scaling: comparable ranges")
for ax in axes: ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

**Effect:** after pre-processing there are no missing values, and all features have mean 0 and standard
deviation 1 on the training data, so no feature dominates just because of its units. The **shape** of each
distribution does not change (scaling only shifts and stretches), and the order of values is kept.

The pre-processor is built with `make_preprocessor()` and is placed **inside each model's pipeline** in Section 4,
so it is always fitted on training data only. The preprocessing-order experiment (fitting it before vs after the
split) is reported with the models, since it needs a trained model to measure the effect.

# Section 4: Predictive Modeling

Binary classification: **podium (1) vs not (0)**. All models use the **same rows and the same 8 features** from
Section 3, so their scores are directly comparable.

| Model | Role | How it works |
|---|---|---|
| Logistic Regression | **Baseline** (simplest reasonable model) | Adds up the features with one weight each and turns the sum into a probability with the sigmoid function. Only learns straight-line (linear) relationships. |
| Random Forest | Second statistical model | Builds many decision trees on random samples of rows and features, and averages their votes. Can learn non-linear patterns and interactions (e.g. "front row **and** strong team"). |
| Neural Network (MLP) | Shallow neural network | 1–2 hidden layers of neurons with ReLU activation, trained with gradient descent (Adam). Can learn non-linear patterns. |

Every model is a pipeline: **pre-processor from Section 3 (median imputation + scaling) → model**, so the
pre-processor is always fitted on the training data only.

**Metrics** (all five, on train, validation and test): ROC-AUC, PR-AUC, F1, precision, recall.
- **Primary metric: PR-AUC.** Only about 14% of rows are podiums. PR-AUC focuses on how well the model finds the
  rare podium finishers. ROC-AUC can look high even when the model makes many false podium predictions.
- **Secondary metric: ROC-AUC**, to show how well the two classes are separated overall.
- **F1, precision and recall** need a probability **threshold**. We choose it on the **validation** set
  (the threshold that gives the best F1), never on the test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, precision_recall_curve, roc_curve)

def make_model(estimator):
    return Pipeline([("prep", make_preprocessor()), ("model", estimator)])

def best_threshold(y_true, proba):
    # threshold that maximises F1 (used on the validation set only)
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p * r / (p + r + 1e-12)
    return t[np.argmax(f1[:-1])]

def all_metrics(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {"ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba),
            "F1": f1_score(y_true, pred), "precision": precision_score(y_true, pred), "recall": recall_score(y_true, pred)}

## 4.1 Hyperparameter selection (training + validation only)

Each candidate is trained on the **training** seasons and scored on the **validation** seasons with the primary
metric (PR-AUC). The best setting is kept. The test set is not used here.
- **Logistic Regression (baseline):** strength of regularisation `C`.
- **Random Forest:** maximum tree depth (deeper trees fit the training data more closely but can overfit).
- **Neural Network:** size of the hidden layers (one or two layers).

In [ ]:
candidates = {
    "Logistic Regression": {f"C={c}": LogisticRegression(C=c, max_iter=1000) for c in [0.01, 0.1, 1, 10]},
    "Random Forest": {f"max_depth={d}, min_samples_leaf={l}": RandomForestClassifier(n_estimators=300, max_depth=d, min_samples_leaf=l, random_state=42, n_jobs=-1) for d in [4, 6, 8, 12] for l in [5, 20, 50]},
    "Neural Network": {f"hidden={h}": MLPClassifier(hidden_layer_sizes=h, alpha=1e-3, max_iter=500, early_stopping=True,
                                                    random_state=42) for h in [(8), (16), (32), (32, 16)]},
}

tuning_rows, best = [], {}
for name, options in candidates.items():
    for setting, est in options.items():
        m = make_model(est).fit(X_train, y_train)
        val_pr_auc = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        tuning_rows.append({"model": name, "setting": setting, "validation PR-AUC": val_pr_auc})
        if name not in best or val_pr_auc > best[name][2]:
            best[name] = (setting, m, val_pr_auc)

tuning = pd.DataFrame(tuning_rows)
print("Chosen:", {name: b[0] for name, b in best.items()})
tuning.round(4)

## 4.2 Results on train, validation and test

For each chosen model: the threshold is picked on **validation**, then the same threshold is applied to train,
validation and test.

In [ ]:
models = {name: b[1] for name, b in best.items()}
rows, thresholds = [], {}
for name, m in models.items():
    thresholds[name] = best_threshold(y_val, m.predict_proba(X_val)[:, 1])
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
        rows.append({"model": name, "split": split, "threshold": thresholds[name],
                     **all_metrics(y, m.predict_proba(X)[:, 1], thresholds[name])})

results = pd.DataFrame(rows).set_index(["model", "split"])
results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

results.reset_index().pivot(index="model", columns="split", values="PR-AUC")[["train", "validation", "test"]].plot(
    kind="bar", ax=axes[0], rot=0)
axes[0].set(title="PR-AUC (primary metric) by split", ylabel="PR-AUC", ylim=(0, 1))

for name, m in models.items():
    proba = m.predict_proba(X_test)[:, 1]
    p, r, _ = precision_recall_curve(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[1].plot(r, p, label=f"{name} (PR-AUC={average_precision_score(y_test, proba):.3f})")
    axes[2].plot(fpr, tpr, label=f"{name} (ROC-AUC={roc_auc_score(y_test, proba):.3f})")
axes[1].axhline(y_test.mean(), ls="--", color="gray", label=f"random guess ({y_test.mean():.2f})")
axes[1].set(title="Precision-recall curve (test)", xlabel="recall", ylabel="precision")
axes[2].plot([0, 1], [0, 1], ls="--", color="gray", label="random guess")
axes[2].set(title="ROC curve (test)", xlabel="false positive rate", ylabel="true positive rate")
axes[1].legend(fontsize=8); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Training progress of the neural network: training loss and validation score per epoch
# (early_stopping keeps 10% of the TRAINING rows aside as an internal check; our validation/test sets are not used)
nn = models["Neural Network"].named_steps["model"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nn.loss_curve_, label="training loss")
ax2 = ax.twinx()
ax2.plot(nn.validation_scores_, color="C1", label="internal validation accuracy")
ax.set(title="Neural network training (early stopping)", xlabel="epoch", ylabel="loss")
ax2.set_ylabel("accuracy"); ax2.grid(False)
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc="center right"); plt.tight_layout(); plt.show()
print("Stopped after", nn.n_iter_, "epochs")

**Interpretation:**

- **Hyperparameters (4.1):** chosen on validation PR-AUC:
  - Logistic Regression: `C=1`
  - Random Forest: `max_depth=6, min_samples_leaf=5`. Deeper trees (8, 12) and larger leaves (20, 50) scored *lower* on validation; deeper trees with small leaves overfit (high train score, lower validation score).
  - Neural Network: two hidden layers `(32, 16)`.
- **All three models perform well and very close to each other:**
  - Test PR-AUC is 0.72–0.73, about **5× better than random guessing** (0.15).
  - Test ROC-AUC is 0.93–0.94.
- **Random Forest is the best model on the primary metric:**
  - Validation PR-AUC 0.718 vs 0.703 for the baseline.
  - Test PR-AUC 0.730 vs 0.721.
  - We choose it because it is best on **validation**. The test set was only used for the final check.
- **Comparison with the baseline:**
  - The gain over Logistic Regression is small (about +0.01 PR-AUC).
  - So most of the signal is simple and almost linear: start near the front, with a strong team in good form, and you are likely to finish on the podium.
  - The Random Forest earns its extra complexity only slightly.
  - The Neural Network does not beat the baseline, which is expected with only 8 features and about 6,000 training rows.
- **Overfitting check:**
  - Random Forest scores higher on train (0.756) than on validation/test (0.718/0.730), so it memorises a little of the training data. The gap is small.
  - Logistic Regression and the Neural Network have almost no gap.
- **Generalisation to new seasons:** test scores are as good as or slightly better than validation for every model. The models still work in 2021–2024, including the 2022 regulation change.
- **Threshold (F1, precision, recall):**
  - With the threshold chosen on validation, the Random Forest's test results are **F1 0.67**, precision 0.69 and recall 0.66.
  - In other words, about 2 out of 3 predicted podiums are correct, and about 2 out of 3 real podiums are found.

## 4.3 Limitations of the models

- **Logistic Regression:** only learns linear relationships, so it cannot learn interactions (e.g. a front-row
  start matters more for a strong team) unless we create them by hand. Correlated features (grid vs qualifying
  position, points vs rank) make its individual weights hard to interpret.
- **Random Forest:** can memorise the training data (training scores are higher than validation/test).
  Its probabilities are averages of tree votes and are not always well calibrated. It is slower and harder to
  explain than logistic regression, which is why Section 6 uses permutation importance and SHAP.
- **Neural Network:** results depend on the random starting weights and the chosen architecture; it is a
  "black box" and needs scaled inputs. With only ~6,000 training rows and 8 features there is little room for it
  to beat simpler models.
- **All models:** they only see pre-race information, so they cannot predict crashes, mechanical failures,
  weather or strategy. They are trained on 2003–2017 and tested on 2021–2024, so a change in the sport (e.g. the
  2022 rules) can shift the patterns. Podiums are rare (~14%), so F1 depends strongly on the chosen threshold.

# Section 5: Experiments

## 5.1 Feature-group ablation

We remove **one group of features at a time**, retrain the chosen model (Random Forest from Section 4, **same
hyperparameters**) and measure how much the **validation** score drops. A big drop means the model really needs that
group. Groups:
- **grid**, **qualifying position**, and **all qualifying features** (grid + qualifying position together, because
  the two are 0.97 correlated, see Section 3.3)
- **standings** (points and rank of driver and constructor before the race)
- **form** (driver and constructor podiums in recent races)

In [ ]:
from sklearn.base import clone

final_model = models["Random Forest"]   # chosen model from Section 4 (same hyperparameters)

def validation_scores(cols):
    m = clone(final_model).fit(X_train[cols], y_train)
    proba = m.predict_proba(X_val[cols])[:, 1]
    return average_precision_score(y_val, proba), roc_auc_score(y_val, proba)

ABLATION_GROUPS = {
    "grid": ["grid"],
    "qualifying position": ["quali_position"],
    "all qualifying (grid + qualifying position)": ["grid", "quali_position"],
    "standings": STANDINGS,
    "form": FORM,
}

full_pr, full_roc = validation_scores(FEATURES)
rows = [{"removed group": "none (all 8 features)", "features used": len(FEATURES),
         "validation PR-AUC": full_pr, "validation ROC-AUC": full_roc}]
for group, cols in ABLATION_GROUPS.items():
    kept = [f for f in FEATURES if f not in cols]
    pr, roc = validation_scores(kept)
    rows.append({"removed group": group, "features used": len(kept), "validation PR-AUC": pr, "validation ROC-AUC": roc})

ablation = pd.DataFrame(rows)
ablation["PR-AUC drop"] = full_pr - ablation["validation PR-AUC"]

ax = ablation.iloc[1:].set_index("removed group")["PR-AUC drop"].sort_values().plot(kind="barh", figsize=(9, 4))
ax.axvline(0, color="k", lw=0.8)
ax.set(title="Drop in validation PR-AUC when a feature group is removed", xlabel="PR-AUC drop (bigger = more important)")
plt.tight_layout(); plt.show()
ablation.round(4)

**Interpretation:**

- **The qualifying features matter most.** Removing grid and qualifying position together drops validation PR-AUC from **0.718 to 0.624** (−0.093), by far the largest drop. Where a driver starts is the strongest pre-race signal, which matches the EDA hypothesis in Section 3.3.
- **Removing just one of the two hurts much less:**
  - Without `grid`: −0.030
  - Without `quali_position`: −0.005
  - The two features carry almost the same information (correlation 0.97), so when one is removed the model uses the other.
  - `grid` is the more useful of the two because it includes penalties, so it is closer to where the car really starts (Section 2, Q1).
- **Standings add a little:** −0.014.
- **Form adds the least:** −0.008. Part of what form says (a strong car or driver) is already captured by standings and grid.
- **Every group lowers the score when removed**, so all groups contribute something and none should be dropped. However, the model depends mainly on the starting position.
- ROC-AUC follows the same order but changes less, because it is less sensitive to how well the rare podium class is ranked.

## 5.2 Pre-processing order experiments

Section 3.4 fits the imputer and scaler on the **training data only, after the split**. Here we test what happens
if they are fitted **before the split, on all the data** (train + validation + test), which is a common mistake:

| Run | Imputer (median) fitted on | Scaler fitted on |
|---|---|---|
| Correct (our pipeline) | train only | train only |
| Experiment 1 | **all data** | train only |
| Experiment 2 | train only | **all data** |
| Experiment 3 | **all data** | **all data** |

We use **Logistic Regression** (the baseline), because it depends on scaling. The Random Forest is not affected by
scaling, so it would not show the effect.

In [ ]:
X_all = pd.concat([X_train, X_val, X_test])

def run_order(impute_fit_data, scale_fit_data):
    imp = SimpleImputer(strategy="median").fit(impute_fit_data)
    sc = StandardScaler().fit(imp.transform(scale_fit_data))
    prep = lambda X: sc.transform(imp.transform(X))
    lr = LogisticRegression(C=1, max_iter=1000).fit(prep(X_train), y_train)
    out = {}
    for split, X, y in [("validation", X_val, y_val), ("test", X_test, y_test)]:
        proba = lr.predict_proba(prep(X))[:, 1]
        out[f"{split} PR-AUC"] = average_precision_score(y, proba)
        out[f"{split} ROC-AUC"] = roc_auc_score(y, proba)
    out["median used for driver_rank_before"] = imp.statistics_[FEATURES.index("driver_rank_before")]
    out["mean used for driver_points_before"] = sc.mean_[FEATURES.index("driver_points_before")]
    return out

order_runs = {
    "Correct: both fitted on train only": (X_train, X_train),
    "Exp 1: imputer fitted on all data": (X_all, X_train),
    "Exp 2: scaler fitted on all data": (X_train, X_all),
    "Exp 3: both fitted on all data": (X_all, X_all),
}
order_results = pd.DataFrame({name: run_order(*data) for name, data in order_runs.items()}).T
order_results.round(4)

**Interpretation:**

- **Fitting the pre-processing on all the data makes almost no difference to the scores here:** validation and test PR-AUC change by at most 0.0003.
- **Why so small:**
  - The **median** used to fill missing values is the same (11.0 for `driver_rank_before`) whether it comes from the training data or from all the data, so Experiment 1 gives identical results.
  - The **scaler** does change. The mean of `driver_points_before` goes from **30.7** (train only) to **38.2** (all data), because later seasons award more points. But scaling only shifts and stretches each feature, and Logistic Regression adjusts its weights to compensate, so the predictions barely change.
- **We still fit on the training data only.** Experiments 2 and 3 show that fitting on all the data **uses information from the validation and test seasons** (the mean changed from 30.7 to 38.2). That is leakage, even when it does not visibly change the score. In a dataset with bigger differences between periods, or with more missing values, it could make the results look better than they really are.
- **Conclusion:** the correct order is **split first, then fit the pre-processing on the training data only**. Our pipeline in Sections 3.4 and 4 does this by putting the imputer and scaler inside each model's `Pipeline`.

# Section 6: Model Explainability (XAI)

We explain the **chosen model (Random Forest from Section 4)** on the **unseen test seasons (2021–2024)**:
- **Global explanations** (how the model behaves overall): **permutation importance** and **SHAP**.
- **Local explanation** (one single prediction): a **SHAP waterfall plot** for one driver in one race.

> **Important: these explanations are not causal.** A feature being important only means the model **relies on it
> to make predictions**. It does **not** prove that the feature **causes** podium finishes. For example, a good grid
> position and a podium are both results of having a fast car and a good driver.

## 6.1 Global: permutation importance

For each feature we **randomly shuffle its values** in the test set and measure how much the model's PR-AUC drops.
If the score drops a lot, the model relies on that feature. This is repeated 10 times per feature.

In [ ]:
from sklearn.inspection import permutation_importance

final_model = models["Random Forest"]
perm = permutation_importance(final_model, X_test, y_test, scoring="average_precision", n_repeats=10, random_state=42)
perm_df = pd.DataFrame({"feature": FEATURES, "mean PR-AUC drop": perm.importances_mean,
                        "std": perm.importances_std}).sort_values("mean PR-AUC drop")

ax = perm_df.plot(kind="barh", x="feature", y="mean PR-AUC drop", xerr="std", legend=False, figsize=(9, 4))
ax.axvline(0, color="k", lw=0.8)
ax.set(title="Permutation importance (test set, Random Forest)", xlabel="drop in PR-AUC when the feature is shuffled")
plt.tight_layout(); plt.show()
perm_df.sort_values("mean PR-AUC drop", ascending=False).round(4)

**Interpretation:**

- **The model relies mostly on starting position:**
  - Shuffling `grid` lowers test PR-AUC by **0.118**.
  - Shuffling `quali_position` lowers it by **0.070**.
- **All other features matter much less** (≤ 0.013 each). Driver rank and the two form features come next.
- **Why the points features score lowest:** they carry almost the same information as rank and form (correlations up to 0.94, Section 3.3). When one of them is shuffled, the model still gets that information from the others, so its importance looks small. Low permutation importance therefore does not mean the information is useless, only that it is duplicated.
- **Why this differs from the ablation (Section 5.1), where removing grid alone cost only 0.030:**
  - Permutation importance keeps the trained model **fixed**, so the model cannot replace grid with qualifying position.
  - In the ablation the model is **retrained** and switches to `quali_position` instead.
  - Both agree that starting position is the most important information.

## 6.2 Global: SHAP

**SHAP** splits each prediction into a contribution from every feature: positive values push the prediction
**towards podium**, negative values push it **away from podium**. Averaging the size of these contributions over all
test rows gives a global importance; the beeswarm plot also shows the **direction** (colour = feature value).

In [ ]:
import shap

prep = final_model.named_steps["prep"]
rf = final_model.named_steps["model"]
X_test_model = prep.transform(X_test)                                   # what the model sees (imputed + scaled)
X_test_display = pd.DataFrame(prep.named_steps["impute"].transform(X_test), columns=FEATURES, index=X_test.index)  # readable values

explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_test_model)
sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)      # contributions to class 1 (podium)
base_value = explainer.expected_value[1] if np.ndim(explainer.expected_value) else explainer.expected_value

plt.figure()
shap.summary_plot(sv, X_test_display, feature_names=FEATURES, plot_type="bar", show=False)
plt.title("SHAP: mean |contribution| per feature (test set)"); plt.tight_layout(); plt.show()

plt.figure()
shap.summary_plot(sv, X_test_display, feature_names=FEATURES, show=False)
plt.title("SHAP beeswarm: direction of each feature's effect (test set)"); plt.tight_layout(); plt.show()

**Interpretation:**

- **SHAP gives the same order as permutation importance:** grid and qualifying position first, then driver rank, constructor form, driver form, constructor rank, and the points features last. Two different methods agreeing makes the result more reliable.
- **Direction of the effects (beeswarm, blue = low value, red = high value):**
  - **grid / quali_position:** low values (starting near the front) push the prediction **towards podium**, by up to +0.3. High values push it away.
  - **driver_rank_before / constructor_rank_before:** a good rank (low number, blue) pushes **towards podium**.
  - **driver_form / constructor_form:** recent podiums (red) push **towards podium**.
  - **points:** more points (red) push slightly **towards podium**.
- **These directions match the EDA in Section 3.3.** The model has learned sensible patterns rather than anything strange.
- **Reminder:** these are effects on the **model's prediction**, not effects on real race results (see the note at the top of Section 6).

## 6.3 Local: explaining one prediction (SHAP waterfall)

We pick one test row where the driver did **not** start in the top 3, but the model still predicted a **podium**,
and show which features pushed the prediction up or down.

In [ ]:
proba_test = final_model.predict_proba(X_test)[:, 1]
cand = np.where((X_test["grid"].values > 3) & (proba_test >= thresholds["Random Forest"]))[0]
i = cand[0]
row = test.iloc[i]
driver = clean["drivers"].set_index("driverId").loc[row.driverId]
race = clean["races"].set_index("raceId").loc[row.raceId]
print(f"{driver.forename} {driver.surname}, {race['name']} {int(race.year)} (round {int(race['round'])})")
print(f"Started P{int(row.grid)} | predicted podium probability = {proba_test[i]:.2f} "
      f"(threshold {thresholds['Random Forest']:.2f}) | actual result: P{int(row.positionOrder)}")

explanation = shap.Explanation(values=sv[i], base_values=base_value, data=X_test_display.iloc[i].values,
                               feature_names=FEATURES)
plt.figure()
shap.plots.waterfall(explanation, show=False)
plt.title("Why the model predicted this row"); plt.tight_layout(); plt.show()

**Interpretation (Lewis Hamilton, Monaco GP 2021, started P7):**

- **Starting point:** the model's average prediction is **0.14** (the podium rate). Each feature then moves it up or down, ending at **0.48**.
- **Pushed up:**
  - Team form: Mercedes averaged 1.75 podiums per race recently (**+0.12**).
  - Hamilton was **leading the championship** (rank 1, **+0.12**).
  - He had a podium in every race of 2021 so far (driver form 1.0, **+0.10**).
  - Mercedes led the constructors' championship (**+0.04**).
- **Pulled down:** starting **P7** on the grid (**−0.04**) and qualifying 7th (**−0.03**).
- **Result:** 0.48 is just above the threshold (0.45), so the model predicted a **podium**. In reality Hamilton **finished P7**, so this is a **wrong prediction (false positive)**.
- **Why it went wrong:** the model trusted Hamilton's and Mercedes' strength, but it does not know the **circuit**. Monaco is famously hard to overtake on (Section 2, Q1 showed Monaco near the top for front-row conversion), so starting P7 there is worse than starting P7 at most tracks. This illustrates the limitation noted in Section 4.3.
- **Not causal:** the waterfall shows how the **model** reached its prediction. It does not mean that leading the championship *causes* podiums. It only means the model uses it as a signal.

# Section 7: Inference Function

`predict_podium(record)` predicts whether **one driver** will finish on the podium in **one race**, using only
information available **after qualifying and before the race**.

- **Input:** a dictionary with the 8 features from Section 3. Missing features (left out or `None`) are allowed.
- **How it works:** it uses the final model from Section 4 (Random Forest pipeline). Missing values are filled
  with the **training median**, exactly as during training, then the probability is compared with the threshold
  chosen on the **validation** set.
- **Output:** which features were missing, the podium probability, and the prediction (PODIUM / NO PODIUM).

We demonstrate it on a **complete record** and a **partially-missing record**.

In [ ]:
def predict_podium(record, model=final_model, threshold=thresholds["Random Forest"]):
    """Predict a podium for ONE driver in ONE race, using only pre-race information.
    record: dict with the 8 features from Section 3. Missing features (absent or None) are allowed:
    the pipeline fills them with the training median, exactly as during training."""
    row = pd.DataFrame([{f: record.get(f) for f in FEATURES}]).astype(float)   # absent / None -> NaN
    proba = model.predict_proba(row)[0, 1]
    return {"missing features": [f for f in FEATURES if pd.isna(row.loc[0, f])],
            "podium probability": round(float(proba), 3),
            "prediction": "PODIUM" if proba >= threshold else "NO PODIUM"}

# 1) Complete record: Max Verstappen, 2023 round 10 (a real row from the unseen test set)
example = test[(test.driverId == 830) & (test.year == 2023) & (test["round"] == 10)].iloc[0]
complete_record = example[FEATURES].to_dict()
print("Complete record:", complete_record)
print("->", predict_podium(complete_record), "| actual finish: P" + str(int(example.positionOrder)))

# 2) Partially-missing record: a season opener (round 1) - grid and qualifying are known, but there are no
#    standings ranks or form yet, so those four features are missing (not even in the dict)
partial_record = {"grid": 2, "quali_position": 2, "driver_points_before": 0, "constructor_points_before": 0}
print("\nPartially-missing record:", partial_record)
print("->", predict_podium(partial_record))

**Interpretation:**

- **Complete record** (Verstappen, 2023 round 10): pole position, leading both championships, podium in every
  recent race. The model gives a **0.87** podium probability → **PODIUM**. He actually **won** the race.
- **Partially-missing record** (an example season opener, starting P2): rank and form do not exist yet in round 1,
  so these **4 features are missing**. The function still works: the pipeline fills them with the training
  medians, and the model relies on the known features (grid and qualifying P2). It gives **0.55** → **PODIUM**.
- The second probability is **lower and less certain** than the first, because without standings and form the
  model only knows the starting position. This matches Section 5, where starting position was the most important
  group but not the only one.
- Missing values are handled **inside the model's pipeline**, using medians from the training data only, so the
  function behaves exactly like the evaluation in Section 4 and cannot leak information from the future.